In [1]:
import pandas as pd

In [2]:
df = pd.read_json('../data/민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 38966 entries, 0 to 38965
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   도메인        38966 non-null  str  
 1   카테고리       38966 non-null  str  
 2   대화셋일련번호    38966 non-null  str  
 3   화자         38966 non-null  str  
 4   문장번호       38966 non-null  int64
 5   고객의도       38966 non-null  str  
 6   상담사의도      38966 non-null  str  
 7   QA         38966 non-null  str  
 8   고객질문(요청)   38966 non-null  str  
 9   상담사질문(요청)  38966 non-null  str  
 10  고객답변       38966 non-null  str  
 11  상담사답변      38966 non-null  str  
 12  개체명        38966 non-null  str  
 13  용어사전       38966 non-null  str  
 14  지식베이스      38966 non-null  str  
dtypes: int64(1), str(14)
memory usage: 4.5 MB


In [4]:
df.head(10)

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,1,버스노선,,Q,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,,,,"서울, 가산동, 남대문시장, 버스, 노선",서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단,"가산동,교통수단"
1,다산콜센터,대중교통 안내,B2033,상담사,2,,버스노선,Q,,가산동 어디에서 출발하십니까?,,,"가산동, 출발",가산동/동네/ 출발/출발지,"출발,출발지"
2,다산콜센터,대중교통 안내,B2033,고객,3,버스노선,,A,,,가산동 주민센터입니다.,,"가산동, 주민센터",가산동/동네/ 주민센터/공공기관,"주민센터,공공기관"
3,다산콜센터,대중교통 안내,B2033,상담사,4,,버스노선,A,,,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.,"가산동, 주민센터, 남대문시장,버스, 노선",가산동/동네/ 주민센터/공공기관/ 남대문시장/지명/ 버스/교통수단,"주민센터,교통수단"
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장
5,다산콜센터,대중교통 안내,B2033,상담사,6,,버스정류장,A,,,,문성초등학교 정류장에서 탑승하시면 됩니다.,"문성초등학교, 정류장",문성초등학교/공공기관,"정류장,공공기관"
6,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
7,다산콜센터,대중교통 안내,B2033,상담사,8,,버스요금,A,,,,1200원 입니다.,1200원,1200원/금액,"1200원,금액"
8,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,,"서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
9,다산콜센터,대중교통 안내,B2034,상담사,2,,버스노선,A,,,,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다.","서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"


#### **문제**

1. 일반 행정 데이터와 대중교통 데이터를 로드
2. 두개의 데이터를 단순행결합
3. 데이터의 필터링
    - 고객의 질문에서 즉각적으로 상담사의 답변이 오는 데이터만 필터
    - 고객의 질문이 존재 → 다음 행의 상담사 답변이 존재하는가?
        - 비어 있는 구간의 데이터 형태를 일반화
        - '', ' ', '   ' 등 일반화: strip()
4. 질문 중 중복 데이터를 제거
5. 고객의 질문과 상담사의 답변이 하나의 행이 되도록 작업
6. 완료가 된 DataFrame을 csv로 저장 (민원 질의응답(즉답형데이터).csv)
7. 질문들을 모아서 토큰화(Komoran.morphs()), 벡터화(TF-IDF) 작업
8. 질문 목록 생성
    - 여권 재발급 신청 방법을 알려주세요
    - 전입 신고가 인터넷으로 가능한가요?
    - 지방세 환급금을 어디서 신청하나요?
    - 유사한 질문과 답변을 출력(2개씩)

**1**

In [9]:
df_normal = pd.read_json(r'..\data\민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json')
df_traffic = pd.read_json(r'..\data\민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json')

In [13]:
df_normal

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50331,다산콜센터,일반행정 문의,B35809,상담사,16,,여성전용아파트,A,,,,"입주신청서와 추천서, 급여내역서 외 기타 서류를 갖고 홈페이지에서 신청하면 됩니다.","입주, 신청서, 추천서, 급여내역서, 서류, 홈페이지","입주/이사, 신청서, 추천서, 급여내역서, 서류/문서, 홈페이지/웹페이지/인터넷","신청서,인터넷"
50332,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,,"입주, 순위","입주/이사, 순위/순서","순위,순서"
50333,다산콜센터,일반행정 문의,B35809,상담사,18,,여성전용아파트,A,,,,1~3순위가 있습니다.,순위,순위/순서,"순위,순서"
50334,다산콜센터,일반행정 문의,B35809,고객,19,여성전용아파트,,Q,1순위는 누가 되나요?,,,,순위,순위/순서,"순위,순서"


In [14]:
df_traffic

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,1,버스노선,,Q,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,,,,"서울, 가산동, 남대문시장, 버스, 노선",서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단,"가산동,교통수단"
1,다산콜센터,대중교통 안내,B2033,상담사,2,,버스노선,Q,,가산동 어디에서 출발하십니까?,,,"가산동, 출발",가산동/동네/ 출발/출발지,"출발,출발지"
2,다산콜센터,대중교통 안내,B2033,고객,3,버스노선,,A,,,가산동 주민센터입니다.,,"가산동, 주민센터",가산동/동네/ 주민센터/공공기관,"주민센터,공공기관"
3,다산콜센터,대중교통 안내,B2033,상담사,4,,버스노선,A,,,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.,"가산동, 주민센터, 남대문시장,버스, 노선",가산동/동네/ 주민센터/공공기관/ 남대문시장/지명/ 버스/교통수단,"주민센터,교통수단"
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38961,다산콜센터,대중교통 안내,B25059,상담사,16,,시내버스 시간,A,,,,네 질문 해 주세요.,질문,질문/문의,"질문,문의"
38962,다산콜센터,대중교통 안내,B25059,고객,17,시내버스 시간,,Q,그럼 순천 시내버스 막차는 몇 시 인가요?,,,,"순천, 시내, 버스, 차, 시","순천/지역, 버스/대중교통, 차/자동차, 시/시간","시내,시간"
38963,다산콜센터,대중교통 안내,B25059,상담사,18,,시내버스 시간,A,,,,노선마다 다르지만 순천 시내버스는 가장 늦는게 23시55분 입니다.,"노선, 순천, 시내, 버스, 시, 분","노선/차편, 순천/지역, 버스/대중교통, 시/시간, 분/시간","순천,시간"
38964,다산콜센터,대중교통 안내,B25059,고객,19,시내버스 시간,,A,,,아 그렇군요. 도움이 많이 됐습니다. 감사합니다.,,도움,도움/상담,"도움,상담"


**2**

In [16]:
df = pd.concat([df_traffic, df_normal], ignore_index = True)

**3**

In [ ]:
flag1 = (df['화자'] == '고객') & (df['QA'] == 'Q')
flag2 = (df['화자'] == '상담사') & (df['QA'] == 'A')
question_index = df.loc[flag1].index
answer_index = df.loc[flag2].index
index_list = []

for i in question_index:
    if (i+1) in answer_index:
        index_list.append(i)
        index_list.append(i+1)

In [67]:
filtered_df = df[df.index.isin(index_list)]

In [63]:
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 51060 entries, 4 to 89301
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   도메인        51060 non-null  str  
 1   카테고리       51060 non-null  str  
 2   대화셋일련번호    51060 non-null  str  
 3   화자         51060 non-null  str  
 4   문장번호       51060 non-null  int64
 5   고객의도       51060 non-null  str  
 6   상담사의도      51060 non-null  str  
 7   QA         51060 non-null  str  
 8   고객질문(요청)   51060 non-null  str  
 9   상담사질문(요청)  51060 non-null  str  
 10  고객답변       51060 non-null  str  
 11  상담사답변      51060 non-null  str  
 12  개체명        51060 non-null  str  
 13  용어사전       51060 non-null  str  
 14  지식베이스      51060 non-null  str  
dtypes: int64(1), str(14)
memory usage: 6.2 MB


**4**: 나는 행작업을 먼저한다

In [68]:
filtered_df.reset_index(drop=True, inplace = True)

In [77]:
filtered_df['상담사답변'] = filtered_df['상담사답변'].shift(-1)

In [78]:
filtered_df

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,문성초등학교 정류장에서 탑승하시면 됩니다.,정류장,,정류장
1,다산콜센터,대중교통 안내,B2033,상담사,6,,버스정류장,A,,,,,"문성초등학교, 정류장",문성초등학교/공공기관,"정류장,공공기관"
2,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
3,다산콜센터,대중교통 안내,B2033,상담사,8,,버스요금,A,,,,,1200원,1200원/금액,"1200원,금액"
4,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다.","서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51055,다산콜센터,일반행정 문의,B35809,상담사,16,,여성전용아파트,A,,,,,"입주, 신청서, 추천서, 급여내역서, 서류, 홈페이지","입주/이사, 신청서, 추천서, 급여내역서, 서류/문서, 홈페이지/웹페이지/인터넷","신청서,인터넷"
51056,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,1~3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"
51057,다산콜센터,일반행정 문의,B35809,상담사,18,,여성전용아파트,A,,,,,순위,순위/순서,"순위,순서"
51058,다산콜센터,일반행정 문의,B35809,고객,19,여성전용아파트,,Q,1순위는 누가 되나요?,,,국가유공자 자녀 및 국민기초생활수급권자를 1순위로 두고있습니다.,순위,순위/순서,"순위,순서"


**5**: 여기서 중복 데이터 제거

In [81]:
filtered_df.drop_duplicates('고객질문(요청)', inplace=True)

**6**

In [83]:
filtered_df.to_csv('민원.csv', index=False)

**7**

In [85]:
new_df = pd.read_csv('민원.csv')
new_df

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,NaN,Q,어느정류장에서 타야합니까?,NaN,NaN,문성초등학교 정류장에서 탑승하시면 됩니다.,정류장,NaN,정류장
1,다산콜센터,대중교통 안내,B2033,상담사,6,NaN,버스정류장,A,NaN,NaN,NaN,NaN,"문성초등학교, 정류장",문성초등학교/공공기관,"정류장,공공기관"
2,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,NaN,Q,버스요금은 얼마입니까?,NaN,NaN,1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
3,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,NaN,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,NaN,NaN,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다.","서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
4,다산콜센터,대중교통 안내,B2034,고객,3,버스시간,NaN,Q,시간은 얼마정도 걸립니까?,NaN,NaN,약 1시간 10분정도 걸립니다.,시간,NaN,시간
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19055,다산콜센터,일반행정 문의,B35809,고객,9,여성전용아파트,NaN,Q,꼭 서울시에 근무해야하나요?,NaN,NaN,서울소재 직장근무로 제한하고있습니다.,"서울시, 근무","서울시/서울/서울특별시, 근무",근무
19056,다산콜센터,일반행정 문의,B35809,고객,11,여성전용아파트,NaN,Q,임대료는 어떻게 되나요?,NaN,NaN,62400원 입니다.,임대료,"임대료, 월세","임대료,월세"
19057,다산콜센터,일반행정 문의,B35809,고객,13,여성전용아파트,NaN,Q,보증금도 있나요?,NaN,NaN,1423200원 입니다.,보증금,보증금/예치금,"보증금,예치금"
19058,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,NaN,Q,입주순위도 있나요?,NaN,NaN,1~3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"


In [88]:
flag4 = (filtered_df.index % 2 != 1)
filtered_df.loc[flag4]

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,문성초등학교 정류장에서 탑승하시면 됩니다.,정류장,,정류장
2,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
4,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다.","서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
6,다산콜센터,대중교통 안내,B2034,고객,3,버스시간,,Q,시간은 얼마정도 걸립니까?,,,약 1시간 10분정도 걸립니다.,시간,,시간
10,다산콜센터,대중교통 안내,B2034,고객,7,버스요금,,Q,버스 요금은 얼마입니까?,,,교통카드로 1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51048,다산콜센터,일반행정 문의,B35809,고객,9,여성전용아파트,,Q,꼭 서울시에 근무해야하나요?,,,서울소재 직장근무로 제한하고있습니다.,"서울시, 근무","서울시/서울/서울특별시, 근무",근무
51050,다산콜센터,일반행정 문의,B35809,고객,11,여성전용아파트,,Q,임대료는 어떻게 되나요?,,,62400원 입니다.,임대료,"임대료, 월세","임대료,월세"
51052,다산콜센터,일반행정 문의,B35809,고객,13,여성전용아파트,,Q,보증금도 있나요?,,,1423200원 입니다.,보증금,보증금/예치금,"보증금,예치금"
51056,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,1~3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"
